##**Bankruptcy Prediction Model**
Built predictive models using econometric indicators to estimate bankruptcy risk. Performed feature engineering, model tuning, and evaluation to achieve strong leaderboard performance.

**Comparison of Final Submission Models**

Over the course of the project, we converged on two final submission strategies:
A single LightGBM model trained with early stopping and then refit on the full training set
A stacked and blended ensemble built from six base models (LightGBM, CatBoost, XGBoost variants) with adversarial threshold search

**1. LightGBM single model**
LightGBM (Light Gradient Boosting Machine) is a fast, high-performance machine learning algorithm developed by Microsoft.
It uses decision trees and gradient boosting to make accurate predictions and is especially good for large datasets and tabular data.
LightGBM is popular because it trains quickly, uses less memory, and often achieves high accuracy with minimal tuning.
The LightGBM model was trained on a stratified train validation split with early stopping on the validation AUC. Training stopped at iteration 1105, where the validation performance peaked:

*Validation AUC: 0.89553*

*Accuracy: 0.955*

*F1 score: 0.1346*

**Confusion matrix:**

TN = 1903, FP = 2
FN = 88, TP = 7

The classification report confirms that the model is very strong at identifying the majority class and still manages to pick up a portion of the rare positive (bankruptcy) cases:
Class 0: precision 0.96, recall 1.00, F1 0.98
Class 1: precision 0.78, recall 0.07, F1 0.13
Using this configuration, retraining on the full dataset and submitting probability predictions for the test set resulted in the following Kaggle scores:

**Public leaderboard AUC: 0.90500**
**Private leaderboard AUC: 0.91191**

These scores make the LightGBM model the best performing submission overall.



**2. Ensemble model (F25 and F26)**
The ensemble approach started from the F21 base models and added diversity through two additional variants, resulting in six base learners in total:
* Base1: LightGBM full features
* Base2: LightGBM top 25 features
* Base3: CatBoost
* Base4: XGBoost
* Base5: Aggressive LightGBM (less regularization)
* Base6: Conservative CatBoost (more regularization)
Out of fold AUCs for individual base models were in the range 0.8988 to 0.9087. Various ensemble strategies were tested:
* F25: six model weighted ensemble with adversarial threshold search
Six model ensemble OOF AUC: 0.90949
Best F1: 0.49318 at threshold around 0.287
* F26: informed model selection and ensemble composition
Top 4 equal weight ensemble OOF AUC: 0.91425
Top 4 equal weight F1: 0.50945

**Confusion matrix for the best ensemble on the full training set:**

TN = 9251, FP = 276
FN = 217, TP = 256

**Classification report for the best ensemble:**
Class 0: precision 0.9771, recall 0.9710, F1 0.9740
Class 1: precision 0.4812, recall 0.5412, F1 0.5095
This shows that the ensemble is more balanced between classes on the training distribution, with much higher recall and F1 for the minority class compared to the single LightGBM validation split.
However, when evaluated on the hidden Kaggle test set, the ensemble did not translate its stronger out of fold performance into a higher leaderboard score. The best ensemble submission achieved:

*Public leaderboard AUC: 0.89730*

*Private leaderboard AUC: 0.91181*



**FINAL CONCLUSION:**
Although the stacked ensemble showed slightly higher out of fold AUC (up to 0.91425) and much better F1 for the minority class on the training data, it generalized slightly worse to the competition test set than the simpler LightGBM model. The single LightGBM submission achieved **0.91191** private AUC compared to **0.91181** for the ensemble and also scored higher on the public leaderboard (0.90500 versus 0.89730).
Based on these final results, the **LightGBM model is selected as the best overall submission**, offering the strongest AUC performance on both public and private leaderboards while keeping the modeling pipeline relatively simple and robust.

In [ ]:
#Final Model (LightGBM)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, classification_report

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# 1. Load raw train and test data
train = pd.read_csv("bankruptcy_Train.csv")
test = pd.read_csv("bankruptcy_Test_X.csv")
sample_sub = pd.read_csv("bankruptcy_sample_submission.csv")

target_col = "class"

# 2. Remove ID column if present (it is an identifier, not a feature)
if "ID" in train.columns:
    train = train.drop(columns=["ID"])
if "ID" in test.columns:
    test_features = test.drop(columns=["ID"])
else:
    test_features = test.copy()

# 3. Split features and target from training data
X = train.drop(columns=[target_col])
y = train[target_col]

# 4. Standardize features for tree based models and for consistency across models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(test_features)

# 5. Train validation split from scaled features
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Convert to LightGBM dataset format
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val)

# LightGBM training parameters - tuned for good AUC on this dataset
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

# Train LightGBM with early stopping on validation data
model_lgb = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_val],
    num_boost_round=3000,
    callbacks=[lgb.early_stopping(stopping_rounds=200)]
)

# Best iteration safe fallback in case best_iteration is not set
best_iter = model_lgb.best_iteration if model_lgb.best_iteration else 500

# ------------------------------------------------
# Extra evaluation metrics for the validation set
# ------------------------------------------------

# Predict validation probabilities and class labels
val_pred_proba = model_lgb.predict(X_val)
val_pred = (val_pred_proba > 0.5).astype(int)

# Compute standard classification metrics on validation set
auc = roc_auc_score(y_val, val_pred_proba)
accuracy = accuracy_score(y_val, val_pred)
f1 = f1_score(y_val, val_pred)
cm = confusion_matrix(y_val, val_pred)

print("LightGBM AUC:", auc)
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_val, val_pred))

# Retrain LightGBM on the full training data using best iteration
full_train = lgb.Dataset(X_scaled, y)

final_lgb = lgb.train(
    params,
    full_train,
    num_boost_round=best_iter
)

# Predict probabilities on the scaled test features
pred_lgb = final_lgb.predict(X_test_scaled)

# Build submission file using sample submission format
sub_lgb = sample_sub.copy()
sub_lgb["class"] = pred_lgb
sub_lgb.to_csv("submission_lightgbm.csv", index=False)

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1105]	valid_0's auc: 0.895533
LightGBM AUC: 0.8955325321176958
Accuracy: 0.955
F1 Score: 0.1346153846153846
Confusion Matrix:
 [[1903    2]
 [  88    7]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1905
           1       0.78      0.07      0.13        95

    accuracy                           0.95      2000
   macro avg       0.87      0.54      0.56      2000
weighted avg       0.95      0.95      0.94      2000

